<a href="https://colab.research.google.com/github/janyamumy-lab/upskill2568/blob/main/Churn_Analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## วัตถุประสงค์ของชุดคำสั่งนี้

ชุดคำสั่งนี้มีวัตถุประสงค์เพื่อสาธิตการสร้างและประเมินผลโมเดล Machine Learning อย่างง่าย เพื่อทำนายการเลิกใช้บริการ (Churn) ของลูกค้าโทรคมนาคม โดยจะใช้ข้อมูลลูกค้าที่มีอยู่มาฝึกโมเดลให้สามารถคาดการณ์ได้ว่าลูกค้าคนใดมีแนวโน้มที่จะเลิกใช้บริการในอนาคต

### ไลบรารีที่ใช้ในชุดคำสั่งนี้

*   **pandas**: เป็นไลบรารีที่ใช้สำหรับการจัดการข้อมูลในรูปแบบตาราง (DataFrame) ซึ่งคล้ายกับ Excel ทำให้การอ่าน, วิเคราะห์ และจัดเตรียมข้อมูลทำได้ง่ายขึ้น
*   **sklearn (scikit-learn)**: เป็นไลบรารีที่ได้รับความนิยมอย่างมากในด้าน Machine Learning ใช้สำหรับสร้างและฝึกโมเดลต่างๆ เช่น โมเดลการตัดสินใจ (Decision Tree) และ Random Forest รวมถึงมีเครื่องมือสำหรับการแบ่งข้อมูล, ประเมินผลโมเดล และจัดการคุณลักษณะของข้อมูล
    *   `train_test_split`: ฟังก์ชันจาก `sklearn` ที่ช่วยแบ่งข้อมูลออกเป็นชุดฝึก (training set) และชุดทดสอบ (testing set) เพื่อให้เราสามารถฝึกโมเดลด้วยชุดข้อมูลหนึ่ง และทดสอบประสิทธิภาพด้วยชุดข้อมูลอีกชุดหนึ่งที่โมเดลไม่เคยเห็นมาก่อน
    *   `DecisionTreeClassifier`: เป็นโมเดล Machine Learning ประเภทหนึ่งที่สร้างต้นไม้การตัดสินใจเพื่อแยกแยะข้อมูล
    *   `RandomForestClassifier`: เป็นโมเดลที่ใช้หลักการรวมต้นไม้การตัดสินใจหลายๆ ต้นเข้าด้วยกัน (Ensemble Learning) เพื่อให้ได้ผลลัพธ์ที่แม่นยำและเสถียรยิ่งขึ้น
    *   `accuracy_score`, `classification_report`, `confusion_matrix`: ฟังก์ชันสำหรับประเมินประสิทธิภาพของโมเดล
    *   `LabelEncoder`: ฟังก์ชันที่ใช้ในการแปลงข้อมูลที่เป็นข้อความ (เช่น 'Male', 'Female') ให้เป็นตัวเลข เพื่อให้โมเดล Machine Learning สามารถนำไปประมวลผลได้
*   **matplotlib.pyplot**: เป็นไลบรารีสำหรับสร้างกราฟและแผนภูมิ เพื่อช่วยให้เราสามารถแสดงผลข้อมูลและทำความเข้าใจรูปแบบต่างๆ ได้ง่ายขึ้น


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

#โหลดข้อมูล
df = pd.read_csv('https://raw.githubusercontent.com/janyamumy-lab/upskill2568/refs/heads/main/TelecomCustomerChurn.csv')

print(df.shape)
print(df.info())
print(df.head())
print(df.describe())


## การจัดเตรียมคุณลักษณะ (Feature Engineering) และการฝึกโมเดล (Train Model)

ในส่วนนี้ เราจะมาดูขั้นตอนสำคัญในการเตรียมข้อมูลก่อนที่จะนำไปใช้กับโมเดล Machine Learning และการฝึกโมเดลให้เรียนรู้จากข้อมูลของเราครับ

### 1. การจัดเตรียมคุณลักษณะ (Feature Engineering)

*   เป็นกระบวนการในการเลือก, สร้าง, หรือแปลงข้อมูลดิบ (raw data) ให้กลายเป็นคุณลักษณะ (features) ที่โมเดล Machine Learning สามารถนำไปใช้ในการเรียนรู้และทำนายได้ดียิ่งขึ้น คุณลักษณะที่ดีจะช่วยให้โมเดลเข้าใจปัญหาและทำนายได้อย่างแม่นยำ
*   **`le = LabelEncoder()`**: เราเริ่มต้นด้วยการสร้างเครื่องมือ `LabelEncoder` ซึ่งมีหน้าที่แปลงข้อมูลประเภทข้อความ (เช่น 'Male', 'Female') ให้เป็นตัวเลข เพราะโมเดล Machine Learning ส่วนใหญ่ทำงานกับตัวเลขเท่านั้น
*   **`df['gender_encoded'] = le.fit_transform(df['gender'])`**: ตรงนี้เรานำคอลัมน์ `gender` (เพศ) ซึ่งเป็นข้อความ มาแปลงให้เป็นตัวเลข และเก็บไว้ในคอลัมน์ใหม่ชื่อ `gender_encoded` (เช่น 'Female' อาจเป็น 0, 'Male' อาจเป็น 1)
*   **`X = df[['tenure','MonthlyCharges','gender_encoded']]`**: เราเลือกคอลัมน์ที่จะใช้เป็น 'คุณลักษณะ' หรือ 'ตัวแปรต้น' (features) สำหรับโมเดล ซึ่งในที่นี้คือ `tenure` (ระยะเวลาการใช้บริการ), `MonthlyCharges` (ค่าบริการรายเดือน) และ `gender_encoded` ที่เพิ่งแปลงไป ข้อมูล `X` นี้จะเป็น 'อินพุต' ให้กับโมเดล
*   **`y = df['Churn']`**: เรากำหนดคอลัมน์ `Churn` เป็น 'ตัวแปรเป้าหมาย' หรือ 'ตัวแปรตาม' (target variable) ซึ่งเป็นสิ่งที่เราต้องการให้โมเดลทำนายว่าลูกค้าจะ 'Churn' (เลิกใช้บริการ) หรือไม่

### 2. การฝึกโมเดล (Train Model)

*   **`train_test_split(X, y, test_size=0.2, random_state=42)`**: ก่อนจะฝึกโมเดล เราจำเป็นต้องแบ่งข้อมูลออกเป็นสองส่วน:
    *   **ชุดฝึก (Training Set)**: เป็นข้อมูลที่โมเดลจะใช้ในการเรียนรู้และสร้างรูปแบบความสัมพันธ์
    *   **ชุดทดสอบ (Testing Set)**: เป็นข้อมูลที่โมเดลไม่เคยเห็นมาก่อน ใช้สำหรับประเมินว่าโมเดลที่เราสร้างขึ้นมานั้นทำงานได้ดีแค่ไหนกับข้อมูลใหม่ๆ
    *   `test_size=0.2` หมายความว่า 20% ของข้อมูลทั้งหมดจะถูกใช้เป็นชุดทดสอบ และอีก 80% เป็นชุดฝึก
    *   `random_state=42` เป็นค่าที่ช่วยให้เราสามารถแบ่งข้อมูลได้เหมือนเดิมทุกครั้งที่รันโค้ด ทำให้ผลลัพธ์ของการทดลองสามารถทำซ้ำได้
*   **`model = DecisionTreeClassifier(max_depth=5, random_state=42)`**: เราสร้างโมเดล Machine Learning ขึ้นมา ในที่นี้คือ `DecisionTreeClassifier` (โมเดลต้นไม้ตัดสินใจ)
    *   `max_depth=5`: เป็นการจำกัดความลึกสูงสุดของต้นไม้ตัดสินใจ เพื่อไม่ให้โมเดลซับซ้อนเกินไปและป้องกันการเรียนรู้ที่มากเกินไป (Overfitting) ที่อาจทำให้โมเดลทำงานได้ไม่ดีกับข้อมูลใหม่
*   **`model.fit(X_train, y_train)`**: นี่คือขั้นตอนการ 'ฝึก' โมเดล โมเดลจะใช้ข้อมูล `X_train` (คุณลักษณะของชุดฝึก) และ `y_train` (ผลลัพธ์ที่ถูกต้องของชุดฝึก) เพื่อเรียนรู้รูปแบบและสร้างกฎเกณฑ์ในการทำนาย
*   **`y_pred = model.predict(X_test)`**: หลังจากฝึกเสร็จแล้ว เราใช้โมเดลที่ผ่านการฝึกมาทำนายผลลัพธ์ `y_pred` (ค่าที่ทำนาย) จากข้อมูล `X_test` (คุณลักษณะของชุดทดสอบ) ที่โมเดลไม่เคยเห็นมาก่อน
*   **การประเมินผล (Evaluation)**: สุดท้าย เราจะใช้เครื่องมือต่างๆ เช่น `accuracy_score`, `classification_report`, และ `confusion_matrix` เพื่อวัดว่าโมเดลของเราทำนายได้แม่นยำแค่ไหน และมีประสิทธิภาพอย่างไรในการแยกแยะลูกค้าที่ 'Churn' ออกจากลูกค้าที่ไม่ 'Churn'

In [ ]:
#Feature Engineering
le = LabelEncoder()
df['gender_encoded'] = le.fit_transform(df['gender']) #label encoder gender
X = df[['tenure','MonthlyCharges','gender_encoded']]  #features
y = df['Churn']                                       #target

#Split train 80% and test 20%
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#Train model
model = DecisionTreeClassifier(max_depth=5, random_state=42)
model.fit(X_train, y_train)

#Evaluation
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print('Accuracy : ',accuracy)
print('Classification report : ',classification_report(y_test, y_pred))
print('Confusion matrix : ',confusion_matrix(y_test, y_pred))


## การปรับปรุงประสิทธิภาพของโมเดล (Feature Tuning และ Algorithm Change)

ในส่วนนี้ เป็นการทดลองปรับปรุงโมเดลเพื่อหวังว่าจะได้ผลลัพธ์ที่ดีขึ้นกว่าเดิม โดยมีสองแนวทางหลักๆ คือ:

### แนวทางที่ 1: เพิ่มคุณลักษณะ (Feature) ให้กับข้อมูล

*   **ทำไมถึงต้องเพิ่มคุณลักษณะ?** ในการทำนาย โมเดลจะเรียนรู้จากข้อมูลที่เราป้อนให้ ยิ่งข้อมูลที่เราให้มีรายละเอียดที่เกี่ยวข้องกับการทำนายมากเท่าไหร่ โมเดลก็จะมีโอกาสเรียนรู้ได้ดีขึ้นเท่านั้น ในที่นี้ เราได้เพิ่ม `Contract` (สัญญา) และ `PaymentMethod` (วิธีการชำระเงิน) เข้าไปเป็นคุณลักษณะใหม่ เพราะเชื่อว่าสิ่งเหล่านี้อาจส่งผลต่อการตัดสินใจเลิกใช้บริการของลูกค้าได้
*   **วิธีการทำ:** เนื่องจากคุณลักษณะเหล่านี้ (Contract, PaymentMethod) เป็นข้อมูลแบบข้อความ (เช่น 'Month-to-month', 'Electronic check') โมเดล Machine Learning ไม่สามารถประมวลผลข้อมูลที่เป็นข้อความได้โดยตรง จึงต้องใช้ `LabelEncoder` (ที่เราใช้ไปแล้วในส่วนของ `gender`) เพื่อแปลงข้อความเหล่านี้ให้เป็นตัวเลข เพื่อให้โมเดลเข้าใจและนำไปใช้ได้
*   **`X2 = df[['tenure','MonthlyCharges','gender_encoded','Contract_encoded','PaymentMethod_encoded']]`**: ตรงนี้คือการสร้างชุดข้อมูลคุณลักษณะใหม่ (`X2`) ที่รวมเอา `tenure`, `MonthlyCharges`, `gender_encoded` และคุณลักษณะที่เพิ่งเข้ารหัสใหม่คือ `Contract_encoded` และ `PaymentMethod_encoded` เข้าไว้ด้วยกัน

### แนวทางที่ 2: เปลี่ยน Algorithm (โมเดล Machine Learning)

*   **ทำไมถึงต้องเปลี่ยน Algorithm?** โมเดลแต่ละชนิดมีจุดเด่นและจุดด้อยต่างกัน การลองเปลี่ยนโมเดลอาจช่วยให้ได้ผลลัพธ์ที่ดีขึ้นสำหรับปัญหาบางประเภท ในที่นี้ จากเดิมที่เราใช้ `DecisionTreeClassifier` (โมเดลต้นไม้ตัดสินใจ) ซึ่งเป็นโมเดลพื้นฐาน เราได้ลองเปลี่ยนมาใช้ `RandomForestClassifier`
*   **`RandomForestClassifier` คืออะไร?** เป็นโมเดลที่ทรงพลังกว่า `DecisionTreeClassifier` เนื่องจาก `RandomForestClassifier` ไม่ได้ใช้ต้นไม้ตัดสินใจแค่ต้นเดียว แต่จะสร้างต้นไม้ตัดสินใจหลายๆ ต้นมารวมกัน (เหมือนป่าไม้) แล้วให้แต่ละต้นช่วยกันโหวตคำตอบสุดท้าย ทำให้ผลลัพธ์ที่ได้มีความแม่นยำและเสถียรมากขึ้น ลดโอกาสที่จะเกิดการเรียนรู้ที่มากเกินไป (Overfitting) กับข้อมูล
*   **`n_estimators=100`**: คือการกำหนดให้ `RandomForestClassifier` สร้างต้นไม้ตัดสินใจขึ้นมาทั้งหมด 100 ต้น เพื่อนำมารวมกัน

### การวิเคราะห์ความสำคัญของคุณลักษณะ (Feature Importance)

*   **`importances.plot(kind='barh')`**: หลังจากฝึกโมเดล `RandomForestClassifier` แล้ว เราสามารถดูได้ว่าคุณลักษณะใดบ้างที่มีความสำคัญต่อการทำนายมากที่สุด `feature_importances_` จะบอกค่าความสำคัญของแต่ละคุณลักษณะออกมาเป็นตัวเลข และเรานำมาแสดงผลด้วยกราฟแท่งแนวนอน (`bar chart`) เพื่อให้เห็นได้ง่ายว่าคุณลักษณะไหนมีผลมากน้อยเพียงใดต่อการทำนายการเลิกใช้บริการของลูกค้า

In [ ]:
#Feature tuning
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt

#แนวทาง 1: เพิ่ม Feature
df['Contract_encoded'] = le.fit_transform(df['Contract'])
df['PaymentMethod_encoded'] = le.fit_transform(df['PaymentMethod'])
X2 = df[['tenure','MonthlyCharges','gender_encoded','Contract_encoded','PaymentMethod_encoded']]
X2_train,X2_test,y2_train,y2_test = train_test_split(X2,y,test_size=0.2,random_state=42)

#แนวทาง 2: เปลี่ยน algorithm
model_rf = RandomForestClassifier(n_estimators=100,random_state=42)
model_rf.fit(X2_train,y2_train)
y_rf_pred = model_rf.predict(X2_test)

#Feature importance
importances = pd.Series(model_rf.feature_importances_, index=X2.columns).sort_values()
importances.plot(kind='barh')
plt.show()


## การประเมินผลโมเดล (Model Evaluation Metrics)

หลังจากที่เราได้สร้างและฝึกโมเดลแล้ว ขั้นตอนสำคัญต่อมาคือการประเมินว่าโมเดลของเราทำงานได้ดีแค่ไหน การประเมินผลจะช่วยให้เราเข้าใจประสิทธิภาพของโมเดล และสามารถเปรียบเทียบโมเดลต่างๆ เพื่อเลือกโมเดลที่ดีที่สุดได้

### Metric ที่สำคัญในการประเมินผลโมเดล Churn (การทำนายลูกค้าเลิกใช้บริการ)

*   **Accuracy (ความแม่นยำ)**:
    *   เป็นการวัดสัดส่วนของจำนวนการทำนายที่ถูกต้องทั้งหมด (ทั้งทำนายว่า `Churn` ถูกและ `ไม่ Churn` ถูก) เทียบกับจำนวนการทำนายทั้งหมด
    *   **การตีความ**: หากค่า Accuracy สูง หมายความว่าโมเดลทำนายได้ถูกต้องโดยรวมบ่อยครั้ง
    *   **ข้อจำกัด**: Accuracy เพียงอย่างเดียวอาจไม่เพียงพอ โดยเฉพาะในกรณีที่ข้อมูลไม่สมดุล (เช่น จำนวนลูกค้าที่ Churn มีน้อยกว่าลูกค้าที่ไม่ Churn มากๆ) เพราะโมเดลอาจทำนายว่า 'ไม่ Churn' เสมอ ซึ่งทำให้ Accuracy สูง แต่จริงๆ แล้วทำนายลูกค้าที่ Churn ได้ไม่ดีนัก

*   **Classification Report (รายงานการจำแนกประเภท)**:
    *   เป็นรายงานที่ให้ข้อมูลเชิงลึกเกี่ยวกับประสิทธิภาพของโมเดลสำหรับแต่ละคลาส (ในที่นี้คือ 'Churn' และ 'No Churn') โดยจะแสดงค่า Precision, Recall, F1-Score และ Support
    *   **Precision (ความแม่นยำของการทำนายบวก)**:
        *   โมเดลทำนายว่า 'Churn' มีกี่เปอร์เซ็นต์ที่ถูกต้องจริงๆ
        *   **การตีความ**: ค่า Precision สูงหมายความว่าเมื่อโมเดลทำนายว่าลูกค้าจะ Churn เราสามารถเชื่อมั่นได้สูงว่าลูกค้าจะ Churn จริงๆ (ลดการทำนายผิดพลาดว่าเป็น Churn ทั้งที่จริงไม่ Churn)
    *   **Recall (ความสามารถในการตรวจจับบวก)**:
        *   จากลูกค้าที่ Churn จริงๆ ทั้งหมด โมเดลสามารถทำนายได้ถูกต้องกี่เปอร์เซ็นต์
        *   **การตีความ**: ค่า Recall สูงหมายความว่าโมเดลสามารถระบุลูกค้าที่มีแนวโน้มจะ Churn ได้เกือบทั้งหมด (ลดการพลาดลูกค้าที่ Churn จริงๆ)
    *   **F1-Score**:
        *   เป็นค่าเฉลี่ยแบบถ่วงน้ำหนัก (harmonic mean) ระหว่าง Precision และ Recall ใช้เป็นตัวชี้วัดเดียวที่พิจารณาทั้งสองค่า
        *   **การตีความ**: ค่า F1-Score สูงแสดงถึงประสิทธิภาพที่ดีทั้งในด้าน Precision และ Recall

*   **Confusion Matrix **:
    *   เป็นตารางที่แสดงผลการทำนายของโมเดลเทียบกับผลลัพธ์จริงในแต่ละคลาส ช่วยให้เราเห็นภาพรวมของความถูกต้องและข้อผิดพลาดในการจำแนกประเภท
    *   **ส่วนประกอบหลัก**:
        *   **True Positive (TP)**: โมเดลทำนายว่า 'Churn' และลูกค้าก็ 'Churn' จริงๆ (ทำนายถูก)
        *   **True Negative (TN)**: โมเดลทำนายว่า 'ไม่ Churn' และลูกค้าก็ 'ไม่ Churn' จริงๆ (ทำนายถูก)
        *   **False Positive (FP)**: โมเดลทำนายว่า 'Churn' แต่ลูกค้าจริงๆ 'ไม่ Churn' (ทำนายผิดพลาด)
        *   **False Negative (FN)**: โมเดลทำนายว่า 'ไม่ Churn' แต่ลูกค้าจริงๆ 'Churn' (ทำนายผิดพลาดร้ายแรง)
    *   **การตีความ**: เราต้องการให้ค่า TP และ TN สูง และค่า FP และ FN ต่ำ โดยเฉพาะในปัญหา Churn เรามักให้ความสำคัญกับการลด False Negative (FN) เพื่อไม่ให้พลาดลูกค้าที่ Churn จริงๆ ซึ่งจะช่วยให้องค์กรสามารถเข้าถึงลูกค้าเพื่อป้องกันการ Churn ได้ทันท่วงที

การพิจารณา Metric เหล่านี้ร่วมกันจะช่วยให้เรามีข้อมูลเพียงพอในการตัดสินใจว่าโมเดลของเรามีประสิทธิภาพตามที่เราต้องการหรือไม่ และควรปรับปรุงโมเดลในส่วนใดต่อไป

In [ ]:
#Compare results
print('Decision Tree : ',accuracy)
print('Random Forest : ',accuracy_score(y_test, y_rf_pred))
print(classification_report(y2_test,y_rf_pred))